# NS11 Tutorial B - Rich-Get-Richer, MusicLab, and Recommendation Feedback

**Lecture:** NS11 - Power Laws and Rich-Get-Richer Phenomena  
**Reading:** Salganik, Dodds, and Watts (2006)

## Learning goals
- Simulate popularity dynamics with and without **social influence**.
- Quantify how social feedback changes **inequality** and **predictability**.
- Reproduce the logic of the **MusicLab** experiment with parallel worlds.
- Study how ranking-based recommendation can amplify, and exploration can reduce, rich-get-richer effects.

## Outline
1. Quality alone versus social influence.
2. Parallel worlds and the unpredictability of popularity.
3. Recommendation as a popularity amplifier.
4. Exploration as a countermeasure.
5. Exercise and takeaway.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - logits.max()
    probs = np.exp(shifted)
    return probs / probs.sum()


def make_quality_vector(n_items=48, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    qualities = rng.normal(loc=0.0, scale=0.25, size=n_items)
    qualities = (qualities - qualities.min()) / (qualities.max() - qualities.min())
    return qualities


def simulate_market(
    qualities,
    n_consumers=4000,
    beta_quality=1.0,
    alpha_social=0.0,
    ranking_bias=0.0,
    exploration=0.0,
    seed=RANDOM_SEED,
    checkpoints=None,
):
    """Stylized cultural market with quality, social influence, and exposure bias."""
    rng = np.random.default_rng(seed)
    n_items = len(qualities)
    counts = np.zeros(n_items, dtype=int)
    checkpoints = [] if checkpoints is None else sorted(set(checkpoints))
    checkpoint_rows = []

    for t in range(1, n_consumers + 1):
        order = np.argsort(-(counts + 1e-6 * rng.random(n_items)))
        ranks = np.empty(n_items, dtype=int)
        ranks[order] = np.arange(1, n_items + 1)
        exposure = ranks.astype(float) ** (-ranking_bias)

        logits = beta_quality * qualities + alpha_social * np.log1p(counts) + np.log(exposure)
        probs = softmax(logits)
        if exploration > 0:
            probs = (1 - exploration) * probs + exploration / n_items

        choice = rng.choice(n_items, p=probs)
        counts[choice] += 1

        if t in checkpoints:
            for item, downloads in enumerate(counts):
                checkpoint_rows.append({'t': t, 'item': item, 'downloads': int(downloads)})

    snapshot_df = pd.DataFrame(checkpoint_rows)
    return counts, snapshot_df


def quality_rank(qualities, item):
    return int(np.argsort(-qualities).tolist().index(int(item)) + 1)


def market_summary(counts, qualities, label):
    counts = np.asarray(counts, dtype=float)
    top10 = head_tail_share(counts, head_fraction=0.10)
    winner = int(np.argmax(counts))
    return {
        'condition': label,
        'Gini': gini_coefficient(counts),
        'HHI': herfindahl_index(counts),
        'top 10% share': top10['head_share'],
        'mean consumed quality': float(np.dot(normalized_shares(counts), qualities)),
        'winner quality rank': quality_rank(qualities, winner),
        'winner song': winner,
        'unseen songs': int(np.sum(counts == 0)),
    }


def run_parallel_worlds(qualities, label, n_worlds=8, **kwargs):
    summaries = []
    ranks = {}
    counts = {}
    for world in range(n_worlds):
        world_counts, _ = simulate_market(qualities, seed=100 + world, **kwargs)
        order = np.argsort(-world_counts)
        world_ranks = np.empty(len(world_counts), dtype=int)
        world_ranks[order] = np.arange(1, len(world_counts) + 1)
        ranks[f'W {world + 1}'] = world_ranks
        counts[f'W {world + 1}'] = world_counts
        summaries.append(market_summary(world_counts, qualities, f'{label} / W {world + 1}'))
    return pd.DataFrame(summaries), pd.DataFrame(ranks), pd.DataFrame(counts)


def average_pairwise_rank_correlation(rank_df):
    corr = rank_df.corr(method='spearman')
    mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    return corr.where(mask).stack().mean()


---
## 1. Quality alone versus social influence

The MusicLab question is not whether quality matters. It is whether **quality alone** explains final popularity.

We simulate two markets on the same catalogue of songs:
- an **independent** world, where choices depend only on song quality,
- a **social-influence** world, where previous downloads also affect the next choice.


In [ ]:
qualities = make_quality_vector(n_items=48, seed=RANDOM_SEED)
song_labels = [f'Song {i:02d}' for i in range(1, len(qualities) + 1)]

independent_counts, _ = simulate_market(
    qualities,
    beta_quality=1.0,
    alpha_social=0.0,
    ranking_bias=0.0,
    exploration=0.0,
    seed=RANDOM_SEED,
)

social_counts, social_snapshots = simulate_market(
    qualities,
    beta_quality=1.0,
    alpha_social=0.6,
    ranking_bias=0.0,
    exploration=0.0,
    seed=RANDOM_SEED,
    checkpoints=[500, 1000, 1500, 2000, 2500, 3000, 3500, 4000],
)

display(pd.DataFrame([
    market_summary(independent_counts, qualities, 'Independent world'),
    market_summary(social_counts, qualities, 'Social-influence world'),
]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(qualities, independent_counts, color=CATEGORY_PALETTE['blue'], alpha=0.75, label='Independent')
axes[0].scatter(qualities, social_counts, color=CATEGORY_PALETTE['orange'], alpha=0.75, label='Social influence')
style_axis(axes[0], title='Final popularity versus latent quality', xlabel='Latent quality $q_i$', ylabel='Final downloads', legend=True)

top_social = np.argsort(-social_counts)[:5]
traj = social_snapshots[social_snapshots['item'].isin(top_social)].copy()
traj['song'] = traj['item'].map(lambda idx: song_labels[idx])
for song, group in traj.groupby('song'):
    axes[1].plot(group['t'], group['downloads'], marker='o', linewidth=2, label=song)
style_axis(axes[1], title='Top songs in one social-influence world', xlabel='Consumers processed', ylabel='Downloads', legend=True)

plt.show()


**Interpretation**
- In the independent world, quality and popularity stay closely aligned.
- With social influence, early advantages get amplified.
- The strongest songs still tend to do well, but final popularity becomes much more uneven.


---
## 2. Parallel worlds and popularity fragility

The key MusicLab idea is simple: keep the catalogue fixed, but run several parallel worlds.

If popularity is strongly path-dependent, then different worlds can produce different winners even when the underlying quality vector is unchanged.


In [ ]:
indep_summary, indep_ranks, _ = run_parallel_worlds(
    qualities,
    label='Independent',
    n_worlds=8,
    beta_quality=1.0,
    alpha_social=0.0,
    ranking_bias=0.0,
    exploration=0.0,
)

social_summary, social_ranks, _ = run_parallel_worlds(
    qualities,
    label='Social influence',
    n_worlds=8,
    beta_quality=1.0,
    alpha_social=0.6,
    ranking_bias=0.0,
    exploration=0.0,
)

parallel_summary = pd.DataFrame([
    {
        'condition': 'Independent worlds',
        'avg pairwise rank correlation': average_pairwise_rank_correlation(indep_ranks),
        'unique winners': indep_summary['winner song'].nunique(),
        'mean winner quality rank': indep_summary['winner quality rank'].mean(),
        'mean Gini': indep_summary['Gini'].mean(),
    },
    {
        'condition': 'Social-influence worlds',
        'avg pairwise rank correlation': average_pairwise_rank_correlation(social_ranks),
        'unique winners': social_summary['winner song'].nunique(),
        'mean winner quality rank': social_summary['winner quality rank'].mean(),
        'mean Gini': social_summary['Gini'].mean(),
    },
])
display(parallel_summary.round(3))

top_quality = np.argsort(-qualities)[:12]
heatmap_matrix = social_ranks.iloc[top_quality].to_numpy()

plot_heatmap(
    heatmap_matrix,
    x_labels=list(social_ranks.columns),
    y_labels=[song_labels[idx] for idx in top_quality],
    title='Final ranks across social-influence worlds',
    figure_size=(8, 6),
    cmap='Blues_r',
    annotate=True,
    fmt='{:.0f}',
    colorbar=True,
)
plt.show()


**Interpretation**
- The independent worlds stay much more similar.
- Under social influence, some songs move sharply up or down across worlds.
- The lecture point becomes visible computationally: we can often predict that **some hit** will emerge, but not reliably **which hit** will win.


---
## 3. Recommendation as a popularity amplifier

Search and recommendation systems often rank already-popular items more prominently.

We model that with a simple **ranking bias**: higher-ranked songs get more exposure before the next choice is made.


In [ ]:
social_plain_counts, _ = simulate_market(
    qualities,
    beta_quality=1.0,
    alpha_social=0.6,
    ranking_bias=0.0,
    exploration=0.0,
    seed=RANDOM_SEED,
)

ranked_counts, _ = simulate_market(
    qualities,
    beta_quality=1.0,
    alpha_social=0.6,
    ranking_bias=0.4,
    exploration=0.0,
    seed=RANDOM_SEED,
)

display(pd.DataFrame([
    market_summary(social_plain_counts, qualities, 'Social influence only'),
    market_summary(ranked_counts, qualities, 'Social influence + ranked exposure'),
]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_rank_size(social_plain_counts, ax=axes[0], label='Social influence only', color=CATEGORY_PALETTE['blue'])
plot_rank_size(ranked_counts, ax=axes[0], label='Ranked exposure', color=CATEGORY_PALETTE['red'])
style_axis(axes[0], title='Popularity tail under recommendation bias', xlabel='Rank', ylabel='Downloads', legend=True)

plot_cumulative_share(social_plain_counts, ax=axes[1], label='Social influence only', color=CATEGORY_PALETTE['blue'])
plot_cumulative_share(ranked_counts, ax=axes[1], label='Ranked exposure', color=CATEGORY_PALETTE['red'])
style_axis(axes[1], title='Head capture of total downloads', xlabel='Top-ranked songs kept', ylabel='Cumulative share', legend=True)

plt.show()


**Interpretation**
- Ranking bias makes the head much steeper.
- A small set of songs captures a much larger share of total downloads.
- This is the algorithmic version of a rich-get-richer feedback loop.


---
## 4. Exploration as a countermeasure

A simple countermeasure is to keep some exploration in the system: a small probability of exposing items that are not already near the top.

We model that with an **exploration rate** that mixes the popularity-based exposure with a small uniform component.


In [ ]:
explore_counts, _ = simulate_market(
    qualities,
    beta_quality=1.0,
    alpha_social=0.6,
    ranking_bias=0.4,
    exploration=0.05,
    seed=RANDOM_SEED,
)

display(pd.DataFrame([
    market_summary(ranked_counts, qualities, 'Ranked exposure only'),
    market_summary(explore_counts, qualities, 'Ranked exposure + exploration'),
]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(qualities, ranked_counts, color=CATEGORY_PALETTE['red'], alpha=0.75, label='Ranked exposure only')
axes[0].scatter(qualities, explore_counts, color=CATEGORY_PALETTE['green'], alpha=0.75, label='With exploration')
style_axis(axes[0], title='Quality versus final popularity', xlabel='Latent quality $q_i$', ylabel='Final downloads', legend=True)

plot_cumulative_share(ranked_counts, ax=axes[1], label='Ranked exposure only', color=CATEGORY_PALETTE['red'])
plot_cumulative_share(explore_counts, ax=axes[1], label='With exploration', color=CATEGORY_PALETTE['green'])
style_axis(axes[1], title='Exploration reduces head dominance', xlabel='Top-ranked songs kept', ylabel='Cumulative share', legend=True)

plt.show()


**Interpretation**
- Exploration lowers concentration.
- More songs receive at least some attention.
- The core tradeoff is clear: recommendation systems can be optimized for short-run efficiency, or they can preserve a healthier discovery process.


---
## 5. Exercise

Keep the same catalogue and vary the **social influence** strength $\alpha$ under ranked exposure.

1. What happens to inequality as $\alpha$ grows?
2. Does the winner remain one of the highest-quality songs?
3. At what point does the system become strongly path-dependent?


In [ ]:
exercise_rows = []
for alpha in [0.0, 0.3, 0.6, 0.9]:
    counts, _ = simulate_market(
        qualities,
        beta_quality=1.0,
        alpha_social=alpha,
        ranking_bias=0.4,
        exploration=0.0,
        seed=RANDOM_SEED,
    )
    row = market_summary(counts, qualities, f'alpha = {alpha:.1f}')
    exercise_rows.append(row)

display(pd.DataFrame(exercise_rows).round(3))


**Takeaway**
- Rich-get-richer dynamics do not eliminate quality, but they change how quality is translated into popularity.
- Social influence increases **inequality** and reduces **predictability across worlds**.
- Recommendation systems can either amplify that feedback or soften it through exploration and niche exposure.
